
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>

# Demo — Build a Feature Engineering Pipeline

In this demo, we will build a feature engineering pipeline that performs data loading, imputation, transformation, and encoding of categorical features. The pipeline will be applied to training and testing datasets, ensuring consistency in data preprocessing. Finally, we will save the pipeline for future reuse, allowing efficient and reproducible data preparation for machine learning.

**Learning Objectives:**

*By the end of this demo, you will be able to:*

* Build a structured feature engineering pipeline that includes multiple preprocessing steps.
* Encode categorical features using **StringIndexer** and **OneHotEncoder**.
* Impute missing numerical values using `Imputer` (mean for continuous columns, median for the binary `SeniorCitizen` column) and scale the continuous features with `StandardScaler`.
* Assemble transformed numerical and encoded categorical features into a single feature vector.
* Prepare the target column for machine learning by converting it to a numeric label, and check for missing labels.
* Apply the feature engineering pipeline to both training and test datasets.
* Validate that the train and test feature vectors have the same size.
* Save the fitted pipeline to a Unity Catalog volume, reload it, and confirm that it produces the same results.

## REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="border-left: 4px solid #F44336; background: #FFEBEE; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #C62828; font-size: 1.1em;">Select Compute</strong>
<p style="margin: 8px 0 0 0; color: #333;">Before starting this notebook, select the required compute environment listed below.</p>
<ul style="margin: 12px 0 0 16px; color: #333;">
<li><strong>Serverless Compute, Version ML V6</strong> - <a href="https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976D2; text-decoration: underline;" target="_blank">How to select an environment version</a></li>
</ul>
<p style="margin: 8px 0 0 0; color: #333;"><strong>NOTE:</strong> This notebook was <strong>developed and tested using Serverless ML V6</strong>. Other compute options may work but are not guaranteed to behave the same or support all features demonstrated.</p>
</div>
</div>
</div>

## ⚠️ SERVERLESS: RESTART YOUR SESSION IF NEEDED

<div style="border-left: 4px solid #FF9800; background: #FFF3E0; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #E65100; font-size: 1.1em;">Restart the Serverless Session</strong>
<p style="margin: 8px 0 0 0; color: #333;">Because this notebook runs on <strong>Serverless</strong>, you may occasionally need to start a fresh session for state to reset cleanly. To do this, either:</p>
<ul style="margin: 12px 0 0 16px; color: #333;">
<li><strong>Restart the session</strong> — open the compute drop-down (top right) and choose <strong>New session</strong> to begin a new session, <em>or</em></li>
<li><strong>Terminate and start</strong> — <strong>terminate</strong> the current Serverless compute, then <strong>start</strong> it again before re-running the cells.</li>
</ul>
</div>
</div>
</div>

<div style="border-left: 4px solid #F44336; background: #FFEBEE; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #C62828; font-size: 1.1em;">If you hit this error</strong>
<p style="margin: 8px 0 0 0;"><code style="background: #FCE4EC; color: #B71C1C; padding: 2px 6px; border-radius: 3px;">[CONNECT_ML.MODEL_SIZE_OVERFLOW_EXCEPTION] Spark Connect ML error: The fitted or loaded model size is about … bytes. Fit or load a model smaller than 268435456 bytes. SQLSTATE: 54000</code></p>
<p style="margin: 12px 0 0 0; color: #333;"><strong>Fix:</strong> <strong>Restart the session</strong> (compute drop-down → <strong>New session</strong>) <em>or</em> <strong>terminate and start</strong> the Serverless compute, then re-run the notebook cells from the top.</p>
</div>
</div>
</div>

## Classroom Setup
Run the following cell to configure your working environment for this course.

This setup will:
- Initialize the `DA` object (Databricks Academy helper)
- Configure your **default catalog** and **schema**
- Provision any supporting configuration needed for this demo

**NOTE:** The `DA` object is only available in Databricks Academy courses.

In [0]:
%run ../Includes/Classroom-Setup-2.6

**Other Conventions:**

Throughout this demo, we'll refer to the object `DA`. This object, provided by Databricks Academy, contains **variables such as your username, catalog name, schema name, working directory, and dataset locations**. Run the code block below to view these details:

In [0]:
print(f"Username:          {DA.username}")
print(f"Catalog Name:      {DA.catalog_name}")
print(f"Schema Name:       {DA.schema_name}")
print(f"Working Directory: {DA.paths.working_dir}")
print(f"Dataset Location:  {DA.paths.datasets}")

In [0]:
# Release any stale ML model references from previous runs (Serverless ML V6 — 1 GB session cache limit)
clear_ml_cache(globals())


## Data Preparation

Before constructing the feature engineering pipeline, we need to ensure the dataset is consistent and properly formatted. This includes handling data types, addressing missing values, and preparing the dataset for further transformations. The `Telco Customer Churn` dataset will be used for this process.

**Steps in Data Preparation:**
1. Load the dataset into a Spark DataFrame.
1. Split the dataset into training and testing sets.
1. Convert Integer and Boolean columns to Double to ensure compatibility with Spark ML.
1. Handle missing values by identifying and imputing them in:
    - Numeric columns (imputed with `Imputer` in the pipeline)
    - String columns (handled by `StringIndexer` in the pipeline)

### Loading the Dataset
We start by loading the dataset from the specified file path using Spark. Note that we will use `.option("nullValue", " ")` since this particular CSV stores some missing values as a single space, which needs to be read as null. This will allow for proper type casting of the `TotalCharges` column.

> This step ensures that only relevant columns are included for feature engineering and model training.

In [0]:
from pyspark.sql.functions import when, col

# Load dataset with spark
shared_volume_name = 'telco' # From Marketplace
csv_name = 'telco-customer-churn-missing' # CSV file name
dataset_path = f"{DA.paths.datasets.telco}/{shared_volume_name}/{csv_name}.csv" # Full path

telco_df = spark.read.option("nullValue", " ").csv(dataset_path, header="true", inferSchema="true", multiLine="true", escape='"')

# Select columns of interest
telco_df = telco_df.select("gender", "SeniorCitizen", "Partner", "tenure", "InternetService", "Contract", "PaperlessBilling", "PaymentMethod", "TotalCharges", "Churn")

display(telco_df)

### Splitting the Dataset into Training and Testing Sets
Before any step that learns from the data, we split it into training and testing sets using an approximately 80/20 split.
> Since telco_df is a PySpark DataFrame, we will use `randomSplit()`.

In [0]:
train_df, test_df = telco_df.randomSplit([.8, .2], seed=42)

### Transforming the Dataset
To ensure that all numerical and categorical features are compatible with machine learning algorithms, we perform several transformations.

**Convert Integer and Boolean Columns to Double**

- Many machine learning algorithms require numeric input, so we convert all **integer and boolean** columns to **double**.
    > This ensures numerical consistency in the dataset.

In [0]:
from pyspark.sql.types import IntegerType, BooleanType, StringType, DoubleType
from pyspark.sql.functions import col, count, when


# Get a list of integer & boolean columns
integer_cols = [column.name for column in train_df.schema.fields if (column.dataType == IntegerType() or column.dataType == BooleanType())]

# Loop through integer columns to cast each one to double
for column in integer_cols:
    train_df = train_df.withColumn(column, col(column).cast("double"))
    test_df = test_df.withColumn(column, col(column).cast("double"))

**Identifying Missing Values**

Handling missing data is crucial to prevent errors and bias in machine learning models. We first check for missing values in numerical and categorical columns.

- **Find Numeric Columns with Missing Values**

In [0]:
from pyspark.sql.functions import count, when

# Identify numeric columns
num_cols = [c.name for c in train_df.schema.fields if c.dataType == DoubleType()]

# Count missing values in numeric columns
num_missing_values_logic = [count(when(col(column).isNull(), column)).alias(column) for column in num_cols]
row_dict_num = train_df.select(num_missing_values_logic).first().asDict()
num_missing_cols = [column for column in row_dict_num if row_dict_num[column] > 0]

print(f"Numeric columns with missing values: {num_missing_cols}")

- **Find String Columns with Missing Values**

In [0]:
# Identify string columns
string_cols = [c.name for c in train_df.schema.fields if c.dataType == StringType()]

# Count missing values in string columns
string_missing_values_logic = [count(when(col(column).isNull(), column)).alias(column) for column in string_cols]
row_dict_string = train_df.select(string_missing_values_logic).first().asDict()
string_missing_cols = [column for column in row_dict_string if row_dict_string[column] > 0]

print(f"String columns with missing values: {string_missing_cols}")

## Creating a Feature Engineering Pipeline

A **Spark ML Pipeline** chains multiple transformation and estimation steps into a single, reproducible workflow. Rather than applying each step manually - which risks inconsistency between training and test data - a pipeline ensures that the exact same transformations are learned from training data and then applied uniformly to new data.

This is especially important for steps like **imputation** and **scaling**, where the statistics used for transformation (e.g., mean, standard deviation) must be derived from training data only and then applied to the test set. This prevents **data leakage**.

**Our pipeline includes the following steps:**

| Step | Transformer | Purpose |
|------|------------|---------|
| 1 | `StringIndexer` | Convert string categories to numeric indices (categorical branch) |
| 2 | `Imputer` (×2) | Fill missing continuous values with the training-set mean, and the binary `SeniorCitizen` column with the training-set median (numeric branch) |
| 3 | `VectorAssembler` | Combine the continuous columns into a single vector (numeric branch) |
| 4 | `StandardScaler` | Standardize the continuous vector (numeric branch) |
| 5 | `OneHotEncoder` | Convert category indices to sparse binary vectors (categorical branch) |
| 6 | `VectorAssembler` | Combine the scaled continuous vector, the binary `SeniorCitizen` column, and the encoded categorical vectors into one `features` vector |

The pipeline has two branches that work on different columns: steps 1 and 5 handle the **categorical** columns, steps 2–4 handle the **numeric** columns, and step 6 joins them. The binary column `SeniorCitizen` is already 0/1, so it is imputed but **not scaled**, as recommended in the *2.4 Lecture*.

### Step 1 - Encode Categorical Features: StringIndexer and OneHotEncoder

Spark ML algorithms require numeric input. To handle string (categorical) columns, we use a two-step encoding approach:

1. **`StringIndexer`** assigns a stable numeric index for each known category. `handleInvalid="keep"` assigns null values or unseen categories a reserved index instead of failing. The `build_string_indexer_model()` helper (defined in the classroom setup) builds the indexer directly from a fixed label vocabulary with `StringIndexerModel.from_arrays_of_labels()` — this keeps the serialized model small and within the Serverless ML V6 per-model size limit. This is a workaround for that limit: on other compute, you would add a regular `StringIndexer(handleInvalid="keep")` to the pipeline so that it learns the categories from `train_df`. Because the helper sorts each label list, categories are numbered **alphabetically** rather than by frequency, the default behavior covered in *2.3_Lecture_Data_Encoding*. For example, `Contract` maps Month-to-month → `0`, One year → `1`, and Two year → `2`. This order does not change the result: after one-hot encoding, each category still has its own position in the vector.

1. **`OneHotEncoder`** converts each numeric index into a sparse binary vector. By default (`dropLast=True`), the last index is left out because it is implied when all other positions are 0. Here, the last index is the extra one that `StringIndexer` reserves for null or unseen values, so a 3-category column such as `Contract` becomes a vector with 3 positions (for example, index `1.0` → `[0, 1, 0]`). This prevents the model from treating category indices as ordered or continuous numbers.

> We define the categorical columns explicitly here, excluding the target column `Churn`, which we will handle separately.

In [0]:
from pyspark.ml.feature import OneHotEncoder

# Define categorical feature columns (excluding target 'Churn')
categorical_cols = ["gender", "Partner", "InternetService", "Contract", "PaperlessBilling", "PaymentMethod"]

categorical_cols_indexed = [c + "_index" for c in categorical_cols]
ohe_cols = [c + "_ohe" for c in categorical_cols]

# Known label vocabulary for this Telco schema — passed to build_string_indexer_model()
TELCO_LABELS = {
    "gender":           ["Male", "Female"],
    "Partner":          ["No", "Yes"],
    "InternetService":  ["Fiber optic", "DSL", "No"],
    "Contract":         ["Month-to-month", "Two year", "One year"],
    "PaperlessBilling": ["Yes", "No"],
    "PaymentMethod":    ["Electronic check", "Mailed check",
                         "Bank transfer (automatic)", "Credit card (automatic)"],
}

# Build a fixed-vocabulary StringIndexerModel (Serverless ML V6 compatible)
string_indexer_model = build_string_indexer_model(spark, categorical_cols, TELCO_LABELS)

# OneHotEncoder: convert numeric indices to sparse binary vectors
one_hot_encoder = OneHotEncoder(
    inputCols=categorical_cols_indexed,
    outputCols=ohe_cols
)

### Step 2 - Impute Missing Numerical Values

Missing values in numerical columns can cause Spark ML estimators to fail. The `Imputer` estimator replaces missing values with a computed statistic. We use two imputers: the **mean** for the continuous columns (`tenure`, `TotalCharges`), and the **median** for the binary `SeniorCitizen` column. The median of a 0/1 column is its more common value, so the column stays 0 or 1 (a mean such as 0.16 is not a valid value for a yes/no column).

> **Important:** The imputer is fitted on the **training set only**. When included in the pipeline, the mean values are computed from `train_df` during `pipeline.fit(train_df)` and then applied to both `train_df` and `test_df` during `.transform()`. This prevents data leakage from the test set into the training process.

In [0]:
from pyspark.ml.feature import Imputer

# Binary 0/1 columns are imputed with the median (their more common value) and are not scaled
binary_cols = ["SeniorCitizen"]
continuous_cols = [c for c in num_cols if c not in binary_cols]

continuous_missing = [c for c in num_missing_cols if c in continuous_cols]
binary_missing = [c for c in num_missing_cols if c in binary_cols]

# outputCols matches inputCols - missing values are filled in-place
imputer = Imputer(
    inputCols=continuous_missing,
    outputCols=continuous_missing,
    strategy="mean"
)
binary_imputer = Imputer(
    inputCols=binary_missing,
    outputCols=binary_missing,
    strategy="median"   # the median of a 0/1 column is its more common value, so it stays 0 or 1
)

### Step 3 - Assemble and Scale Numerical Features

Before scaling, we combine the **continuous** numerical columns into a single vector using `VectorAssembler`. `StandardScaler` then standardizes each value by subtracting the training mean and dividing by the training standard deviation. In Spark ML, subtracting the mean requires `withMean=True` (the default is `withMean=False`, which only scales), so we set both options explicitly. Centering is fine here because the numerical vector is dense; centering a sparse vector would make it dense and use more memory.

> Scaling ensures that features with large numerical ranges (e.g., `TotalCharges`) do not dominate features with smaller ranges (e.g., `tenure`) in distance-based or gradient-based algorithms.

In [0]:
from pyspark.ml.feature import VectorAssembler, StandardScaler

# Assemble the continuous numerical columns into a single vector
numerical_assembler = VectorAssembler(
    inputCols=continuous_cols,
    outputCol="numerical_assembled"
)

# Scale numerical features to standardize values
numerical_scaler = StandardScaler(
    inputCol="numerical_assembled",
    outputCol="numerical_scaled",
    withMean=True,  # subtract the mean (Spark's default is False)
    withStd=True    # divide by the standard deviation
)

### Step 4 - Assemble the Final Feature Vector

The final `VectorAssembler` combines the scaled continuous features, the unscaled binary `SeniorCitizen` column, and the one-hot encoded categorical vectors into a **single feature vector** called `features`. Spark ML estimators read a column named `features` by default, so using this name means no extra configuration later.

In [0]:
# Assemble scaled continuous + binary + one-hot encoded categorical features into a single vector
feature_cols = ["numerical_scaled"] + binary_cols + ohe_cols

vector_assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

### Step 5 - Build the Pipeline

We now combine all transformation steps into a single `Pipeline`. The order of stages matters - for example, `StringIndexer` must run before `OneHotEncoder`, and the numerical `VectorAssembler` must run before `StandardScaler`.

Once defined, the pipeline can be **fitted once** on training data and **applied many times** to new data, making it reliable and production-ready.

In [0]:
from pyspark.ml import Pipeline

# Define the ordered sequence of pipeline stages.
# string_indexer_model is a single fitted StringIndexerModel that handles all 6
# categorical columns at once (inputCols / outputCols lists). Stage order matters:
#   StringIndexerModel → Imputers → num VectorAssembler → StandardScaler → OHE → final VectorAssembler
stages_list = [string_indexer_model, imputer, binary_imputer, numerical_assembler, numerical_scaler, one_hot_encoder, vector_assembler]

# Instantiate the pipeline
pipeline = Pipeline(stages=stages_list)


## Fit the Pipeline

Fitting the pipeline on the training data runs each **Estimator** stage to learn its parameters (e.g., mean values for `Imputer`, scaling factors for `StandardScaler`). The `StringIndexerModel` was pre-built with a fixed vocabulary, so it acts as a Transformer and applies directly during fit. These learned or configured parameters are stored in the resulting `PipelineModel` and reused consistently when transforming new data.

> **Transformers** (e.g., `VectorAssembler` and the pre-built `StringIndexerModel`) do not learn parameters from data - they are applied directly. `OneHotEncoder`, `Imputer`, and `StandardScaler` are **Estimators**: they learn from `train_df` during fit.

**What Happens During Fitting?**

When we call `.fit(train_df)`, each stage in the pipeline processes the data in sequence:

- **StringIndexerModel** maps each string category to a numeric index. The model was pre-built with a fixed Telco vocabulary using `build_string_indexer_model()`, so it acts as a Transformer during `pipeline.fit()` — no data scan is performed at this stage. This keeps the serialized `PipelineModel` within the Serverless ML V6 per-model size limit.

- **Imputers** compute the **mean** of each continuous column and the **median** of `SeniorCitizen` from `train_df`, and store these values to fill missing data during transformation.

- **VectorAssembler** (numerical) combines the continuous columns into a single vector — no learning required.

- **StandardScaler** computes the **mean and standard deviation** of the assembled continuous vector from training data, used to standardize feature values.

- **OneHotEncoder** learns how many categories each indexed column has from the training data, and uses that to convert indices into sparse binary vectors of a fixed size.

- **VectorAssembler** (final) merges all feature vectors into the single `features` column — no learning required.

> **Note:** Spark Connect ML on Serverless ML V6 enforces a **256 MB per-model size limit**. The `build_string_indexer_model()` and `check_model_size()` helpers (defined in the classroom setup) keep this pipeline within that limit. The [Serverless compute limitations](https://docs.databricks.com/aws/en/compute/serverless/limitations) page lists a lower **100 MB** per-model limit, so aim to keep fitted models well below that size.

In [0]:
# Free any prior pipeline_model from the session cache before re-fitting (Serverless ML V6 — 1 GB session cache limit)
clear_ml_cache(globals(), ["pipeline_model"])

# Fit the pipeline on training data - learns all required statistics
pipeline_model = pipeline.fit(train_df)

In [0]:
# Verify the fitted model stays within the Serverless ML V6 256 MB per-model size limit
check_model_size(pipeline_model, f"{DA.paths.working_dir}/_size_check_tmp")


### Apply the Feature Engineering Pipeline

Once the pipeline is **fitted** to the training data, it can be **applied to any dataset** using `.transform()`.
We apply the pipeline to both:
- **Train Dataset (`train_df`)** → Generates **transformed training features**.
- **Test Dataset (`test_df`)** → Ensures that the same transformations are applied consistently.

The output is a **transformed dataset** containing the final `features` vector, ready for modeling.

In [0]:
# Transform both training and test datasets using the fitted pipeline
train_transformed_df = pipeline_model.transform(train_df)
test_transformed_df = pipeline_model.transform(test_df)

In [0]:
# Show a sample of the transformed feature vectors
train_transformed_df.select("features").show(3, truncate=False)

## Prepare the Target Column

Spark ML models require the **target (label) column to be numeric**. In our dataset, the `Churn` column contains string values - `"Yes"` or `"No"`. We convert these to a numeric representation before passing data to a model.

We use the following mapping:
- `"Yes"` → `1.0` (customer churned)
- `"No"` → `0.0` (customer did not churn)
- anything else → `null`, so that a missing or unexpected value is caught instead of being silently counted as "No"

> This step is applied **after** the pipeline transform because `Churn` was intentionally excluded from the feature engineering pipeline - it is our prediction target, not an input feature.

In [0]:
from pyspark.sql.functions import when, col

# Convert Churn string label to numeric (0.0 = No churn, 1.0 = Churned)
# Unexpected or missing values become null so they can be caught, not silently treated as "No"
def add_label(df):
    return df.withColumn(
        "label",
        when(col("Churn") == "Yes", 1.0)
        .when(col("Churn") == "No", 0.0)
        .otherwise(None)
        .cast("double")
    )

train_prepared_df = add_label(train_transformed_df)
test_prepared_df = add_label(test_transformed_df)

# Display the final prepared training dataset
display(train_prepared_df.select("features", "label"))

### Validate the Prepared Data

Before using the prepared data, check that:
- The target `Churn` is not one of the pipeline's input columns.
- Every row has a feature vector and a label.
- The training and test feature vectors have the **same size**. A mismatch means the two sets were not transformed by the same fitted pipeline.

In [0]:
from pyspark.ml.functions import vector_to_array
from pyspark.sql import functions as F

# The target must never be a pipeline input
assert "Churn" not in num_cols + categorical_cols, "The target column must not be a feature"

sizes = {}
for name, df in [("train", train_prepared_df), ("test", test_prepared_df)]:
    sizes[name] = [r.n for r in df.select(F.size(vector_to_array("features")).alias("n")).distinct().collect()]
    null_features = df.filter(F.col("features").isNull()).count()
    null_labels = df.filter(F.col("label").isNull()).count()
    print(f"{name}: feature vector size = {sizes[name]}, rows without features = {null_features}, rows without a label = {null_labels}")
    assert null_features == 0 and null_labels == 0, f"{name} has rows without features or labels"

assert len(sizes["train"]) == 1 and sizes["train"] == sizes["test"], "Train and test feature vectors differ in size"
print(f"Contract (3 categories) is encoded as a vector of size {train_prepared_df.select('Contract_ohe').first()[0].size}")

## Save and Reuse the Pipeline

Preserving the feature engineering pipeline - including all learned parameters and transformation logic - is essential for maintaining reproducibility, enabling version control, and facilitating collaboration. In this section, we will:

1. **Save the Pipeline:** Save the fitted pipeline model to a versioned folder, **`spark_pipelines/telco_pipeline_v1`**, in your working directory (a Unity Catalog volume). A version in the path lets you keep earlier pipelines when you change the preprocessing.

1. **Explore Loaded Pipeline Stages:** Upon loading the pipeline, inspect its stages to confirm the sequence of transformations that were applied.

1. **Check the Reloaded Pipeline:** Apply the reloaded pipeline to the test set and confirm that it produces exactly the same features as the original.


### Save the Pipeline

In [0]:
# Save the pipeline model with overwrite mode
pipeline_path = f"{DA.paths.working_dir}/spark_pipelines/telco_pipeline_v1"

pipeline_model.write().overwrite().save(pipeline_path)
print(f"Saved model to: {pipeline_path}")

### Load and Use Saved Model

In [0]:
# Load the saved pipeline model
from pyspark.ml import PipelineModel

loaded_pipeline = PipelineModel.load(pipeline_path)

# Show pipeline stages
loaded_pipeline.stages

> **Using Saved Pipeline for Reuse**
>
> Although we already applied the pipeline earlier in this demo, we reload the saved pipeline and apply it again here to illustrate how saved pipelines can be reused in production.

In [0]:
# Use the loaded pipeline to transform the test dataset
reloaded_test_df = loaded_pipeline.transform(test_df)
display(reloaded_test_df)

# The reloaded pipeline must produce exactly the same features as the original
original_features = test_transformed_df.select(vector_to_array("features").alias("f"))
reloaded_features = reloaded_test_df.select(vector_to_array("features").alias("f"))
differences = original_features.exceptAll(reloaded_features).count() + reloaded_features.exceptAll(original_features).count()
print(f"Rows whose features differ after reloading: {differences}")
assert differences == 0, "The reloaded pipeline produced different features"

## Conclusion

In this demo, we built a structured feature engineering pipeline to streamline data preparation for machine learning. The pipeline handled missing value imputation using `Imputer` (mean for continuous columns, median for the binary column), encoded categorical features using fixed-vocabulary `StringIndexerModel` (equivalent role to `StringIndexer`) with `OneHotEncoder`, scaled the continuous features with `StandardScaler` (leaving the binary column unscaled), and assembled all features into a single `features` vector ready for model training.

We also prepared the target column by converting the `Churn` string label (`"Yes"` / `"No"`) to a numeric value (`1.0` / `0.0`) as required by Spark ML models, and validated that every row has a label and that train and test feature vectors have the same size.

By fitting the pipeline on training data only and applying it consistently to both training and test sets, we ensured a reproducible and leakage-free feature transformation process. Saving the pipeline to a versioned path, and checking that the reloaded pipeline gives the same results, enables reliable reuse in future workflows.

> **Next step:** Once your features are prepared, you can get a quick baseline model with **Databricks AutoML**, which trains and compares several models and generates an editable notebook for each trial. Model training is covered in the *Machine Learning Model Development* course.

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://www.databricks.com/legal/privacynotice" target="_blank">Privacy Notice</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Community</a>

